In [1]:
import requests
from dotenv import load_dotenv
import os
import json
import time
from typing import List, Dict

load_dotenv()

api_key = os.getenv("RAPID_API_KEY")

In [2]:
# Load existing listings first, before any API calls
def load_existing_listings(filename: str) -> Dict:
    """Load existing listings with their last update time."""
    if os.path.exists(filename):
        with open(filename, "r", encoding="utf-8") as f:
            return {str(item["id"]): item.get("updated_at") 
                   for item in json.load(f)}
    return {}

existing_listings = load_existing_listings("manhattan_listings.json")

In [4]:
url = "https://streeteasy-api.p.rapidapi.com/rentals/search"

headers = {
    "x-rapidapi-key": api_key,
    "x-rapidapi-host": "streeteasy-api.p.rapidapi.com"
}

areas = "roosevelt-island,all-downtown,all-midtown,all-upper-west-side,all-upper-east-side,all-upper-manhattan"

limit = 500
offset = 0
all_listings = []

while True:
    params = {
        "areas": areas,
        "limit": str(limit),
        "offset": str(offset)
    }
    resp = requests.get(url, headers=headers, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    listings = data.get("listings", [])
    if not listings:
        print("No more listings returned. Done.")
        break

    all_listings.extend(listings)
    print(f"Fetched {len(listings)} at offset {offset}, total so far {len(all_listings)}")

    # Move to the next offset
    next_offset = data.get("pagination", {}).get("nextOffset")
    if next_offset is None:
        break
    # adjust for the API's +501
    offset = int(next_offset) - 1

    # be polite to the API
    time.sleep(0.4)

# Save everything into a JSON file
with open("manhattan_listings.json", "w", encoding="utf-8") as f:
    json.dump(all_listings, f, indent=2)

print(f"Saved {len(all_listings)} listings to manhattan_listings.json")

Fetched 500 at offset 0, total so far 500
Fetched 500 at offset 500, total so far 1000
Fetched 500 at offset 1000, total so far 1500
Fetched 500 at offset 1500, total so far 2000
Fetched 500 at offset 2000, total so far 2500
Fetched 500 at offset 2500, total so far 3000
Fetched 500 at offset 3000, total so far 3500
Fetched 500 at offset 3500, total so far 4000
Fetched 500 at offset 4000, total so far 4500
Fetched 500 at offset 4500, total so far 5000
Fetched 500 at offset 5000, total so far 5500
Fetched 500 at offset 5500, total so far 6000
Fetched 432 at offset 6000, total so far 6432
Saved 6432 listings to manhattan_listings.json


In [5]:
new_or_updated_listings = []
for listing in all_listings:
    listing_id = str(listing["id"])
    if (listing_id not in existing_listings or 
        existing_listings[listing_id] != listing.get("updated_at")):
        new_or_updated_listings.append(listing)

print(f"Found {len(new_or_updated_listings)} new or updated listings")

# Save the changed listings first
if new_or_updated_listings:
    with open("changed_listings.json", "w", encoding="utf-8") as f:
        json.dump(new_or_updated_listings, f, indent=2)
    print(f"Saved {len(new_or_updated_listings)} changed listings to changed_listings.json")

# Then update the main listings file
with open("manhattan_listings.json", "w", encoding="utf-8") as f:
    json.dump(all_listings, f, indent=2)
print(f"Updated manhattan_listings.json with {len(all_listings)} total listings")

Found 1552 new or updated listings
Saved 1552 changed listings to changed_listings.json
Updated manhattan_listings.json with 6432 total listings
